# Questions 7-11 by Caleb Christian

Question 7

In [2]:

import pandas as pd
import numpy as np
from scipy.stats import norm

ames = pd.read_csv("C:\\Users\\chair\\uu_inference\\data\\ames_prices.csv")

air = (ames["Central.Air"] == "Y").astype(int)

print(air.value_counts())
print("Estimated proportion:", air.mean())


Central.Air
1    2734
0     196
Name: count, dtype: int64
Estimated proportion: 0.9331058020477816


In [4]:

R = 5000
rng = np.random.default_rng(42)

boot = []

for i in range(R):
    sample = rng.choice(air, size=len(air), replace=True)
    boot.append(sample.mean())

boot = np.array(boot)

print("Bootstrap:", np.std(boot, ddof=1))


Bootstrap: 0.004692462621398659


In [5]:
#took this from inference notebook
p_hat = air.mean()
standard_error = np.std(boot, ddof=1)

ci_90 = (p_hat - 1.645 * standard_error,
         p_hat + 1.645 * standard_error)

ci_95 = (p_hat - 1.960 * standard_error,
         p_hat + 1.960 * standard_error)

ci_99 = (p_hat - 2.576 * standard_error,
         p_hat + 2.576 * standard_error)

print("90% CI:", ci_90)
print("95% CI:", ci_95)
print("99% CI:", ci_99)


90% CI: (np.float64(0.9253867010355808), np.float64(0.9408249030599823))
95% CI: (np.float64(0.9239085753098403), np.float64(0.9423030287857229))
99% CI: (np.float64(0.9210180183350587), np.float64(0.9451935857605045))


In [6]:
z = (p_hat - 0.90) / standard_error

p_value = 2 * norm.cdf(-abs(z))

print("z-statistic:", z)
print("p-value:", p_value)


z-statistic: 7.055101919578828
p-value: 1.724742589854394e-12


Interpretation:

So it seems like the proportion of houses with central air is 93% roughly. The CI are (0.9254,0.9408) for 90%, (0.9239,0.9423) for 95%, and (0.9210,0.9452) for 95%. We reject the null hypo that p = .90 for all intervals since they do not contain any zeroes.

Question 8

In [7]:

data = ames[["area", "price"]].dropna()

area = np.arcsinh(data["area"].to_numpy())
price = np.arcsinh(data["price"].to_numpy())

r = np.corrcoef(area, price)[0, 1]

print("Correlation:", r)


Correlation: 0.7231661530091357


In [8]:

R = 5000
rng = np.random.default_rng(42)

boot = []

for i in range(R):
    idx = rng.choice(len(area), size=len(area), replace=True)
    boot.append(np.corrcoef(area[idx], price[idx])[0, 1])

standard_error = np.std(boot, ddof=1)

print("Bootstrap SE:", standard_error)


Bootstrap SE: 0.010223900961500759


In [11]:

ci_90 = (r - 1.645 * standard_error, 
         r + 1.645 * standard_error)
ci_95 = (r - 1.96 * standard_error, 
         r + 1.96 * standard_error)
ci_99 = (r - 2.576 * standard_error,
         r + 2.576 * standard_error)

print("90% CI:", ci_90)
print("95% CI:", ci_95)
print("99% CI:", ci_99)


90% CI: (np.float64(0.706347835927467), np.float64(0.7399844700908044))
95% CI: (np.float64(0.7031273071245941), np.float64(0.7432049988936772))
99% CI: (np.float64(0.6968293841323097), np.float64(0.7495029218859617))


In [10]:

z = r / standard_error
p_value = 2 * norm.cdf(-abs(z))

print("z-statistic:", z)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    print(alpha, "Reject H0" if p_value < alpha else "Fail to reject H0")


z-statistic: 70.73289889370982
p-value: 0.0
0.1 Reject H0
0.05 Reject H0
0.01 Reject H0


Looking at our results above, we can see that the corr between the arcsinh of area and the arcsinh of price is about 0.7232 which is a positive correlation. The confidence intervals all exclude zero and the p-value is 0 which means we can reject the null hypothesis. If we resampled area and price sperately, we could not see the correlation between the two variables because the observations were not calculated together.

Question 9

In [12]:
meta = pd.read_csv("C:\\Users\\chair\\uu_inference\\data\\metabric.csv")

survival = meta["Overall Survival (Months)"].dropna()

estimate = survival.mean()

R = 5000
rng = np.random.default_rng(42)

boot = []

for i in range(R):
    sample = rng.choice(survival, size=len(survival), replace=True)
    boot.append(sample.mean())

standard_error = np.std(boot, ddof=1)

print("Mean:", estimate)
print("Bootstrap SE:", standard_error)


Mean: 129.09429138744105
Bootstrap SE: 2.142386022768364


In [14]:
ci_90 = (estimate - 1.645 * standard_error, 
         estimate + 1.645 * standard_error)
ci_95 = (estimate - 1.96 * standard_error, 
         estimate + 1.96 * standard_error)
ci_99 = (estimate - 2.576 * standard_error, 
         estimate + 2.576 * standard_error)

print("90% CI:", ci_90)
print("95% CI:", ci_95)
print("99% CI:", ci_99)

print("Mean:", estimate / 12)
print("90% CI in years:", np.array(ci_90) / 12)
print("95% CI in years:", np.array(ci_95) / 12)
print("99% CI in years:", np.array(ci_99) / 12)

90% CI: (np.float64(125.57006637998708), np.float64(132.618516394895))
95% CI: (np.float64(124.89521478281506), np.float64(133.29336799206703))
99% CI: (np.float64(123.57550499278975), np.float64(134.61307778209235))
Mean: 10.757857615620088
90% CI in years: [10.4641722  11.05154303]
95% CI in years: [10.40793457 11.10778067]
99% CI in years: [10.29795875 11.21775648]


In [16]:
z = (estimate - 120) / standard_error
p_value = 2 * norm.cdf(-abs(z))

print("z-statistic:", z)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    print(alpha, "Reject H0" if p_value < alpha else "Fail to reject H0")


z-statistic: 4.244935922280485
p-value: 2.1865589622301028e-05
0.1 Reject H0
0.05 Reject H0
0.01 Reject H0


Looking at our results above, we can see that the mean recorded survival time is about 10 years. All confidence intervals show that the population mean recorded time is greater than 120 months, since their lower bounds are all above 120. This means 120 months is not a possible value for the population mean based on our intervals. The p-value is about 0.00002, so we reject the null hypothesis at all three significance levels, which agrees with our confidence intervals. However, since some patients are still living, their eventual death times are right-censored, therefore we are estimating the average recorded time to death instead of the actual average lifetime of the patients.

In [17]:
#used notebook plus AI to make this code
data = meta[["Chemotherapy", "Overall Survival Status"]].dropna()
data = data[data["Chemotherapy"].isin(["YES", "NO"])]

data["deceased"] = (data["Overall Survival Status"] == "1:DECEASED").astype(int)

yes = data[data["Chemotherapy"] == "YES"]["deceased"]
no = data[data["Chemotherapy"] == "NO"]["deceased"]

estimate = yes.mean() - no.mean()

R = 5000
rng = np.random.default_rng(42)

boot = []

for i in range(R):
    sample_yes = rng.choice(yes, size=len(yes), replace=True)
    sample_no = rng.choice(no, size=len(no), replace=True)
    boot.append(sample_yes.mean() - sample_no.mean())

standard_error = np.std(boot, ddof=1)

print("YES proportion:", yes.mean())
print("NO proportion:", no.mean())
print("Difference:", estimate)
print("Bootstrap SE:", standard_error)


YES proportion: 0.5209790209790209
NO proportion: 0.5856196783349101
Difference: -0.0646406573558892
Bootstrap SE: 0.03246389295216699


In [19]:
ci_90 = (estimate - 1.645 * standard_error, 
         estimate + 1.645 * standard_error)
ci_95 = (estimate - 1.96 * standard_error, 
         estimate + 1.96 * standard_error)
ci_99 = (estimate - 2.576 * standard_error, 
         estimate + 2.576 * standard_error)

print("90% CI:", ci_90)
print("95% CI:", ci_95)
print("99% CI:", ci_99)

z = estimate / standard_error
p_value = 2 * norm.cdf(-abs(z))

print("\nz-statistic:", z)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    print(alpha, "Reject H0" if p_value < alpha else "Fail to reject H0")


90% CI: (np.float64(-0.1180437612622039), np.float64(-0.011237553449574493))
95% CI: (np.float64(-0.1282698875421365), np.float64(-0.001011427169641893))
99% CI: (np.float64(-0.14826764560067138), np.float64(0.018986330888892983))

z-statistic: -1.9911554492596484
p-value: 0.04646379999493879
0.1 Reject H0
0.05 Reject H0
0.01 Fail to reject H0


Looking at our results above, we can see that the proportion of dead patients who got chemotherapy is about 0.5210 compared to 0.5856 for those who didnt do it, making the difference -0.0646. The 90% and 95% confidence intervals both dont have zero, so we can reject the null hypothesis at the 0.10 and 0.05 significance levels. However, the 99% confidence interval contains a zero, so we fail to reject the null hypothesis at the 0.01 level. Our p-value of 0.0465 agrees with these results. Age at diagnosis and tumor stage could be confounding variables, so we cant really say that chemotherapy caused the difference in deceased proportions.

Question 11

In [20]:

nhanes = pd.read_csv("C:\\Users\\chair\\uu_inference\\data\\nhanes_data_17_18.csv", low_memory=False)

data = nhanes[nhanes["AgeInYearsAtScreening"] >= 18]

sleep = data["SleepHoursWeekdaysOrWorkdays"].dropna().to_numpy()

estimate = sleep.mean()

R = 5000
rng = np.random.default_rng(42)

boot = []

for i in range(R):
    sample = rng.choice(sleep, size=len(sleep), replace=True)
    boot.append(sample.mean())

standard_error = np.std(boot, ddof=1)

print("Mean sleep:", estimate)
print("Bootstrap SE:", standard_error)


Mean sleep: 7.62124248496994
Bootstrap SE: 0.022587204784927532


In [21]:

ci_90 = (estimate - 1.645 * standard_error, 
         estimate + 1.645 * standard_error)
ci_95 = (estimate - 1.96 * standard_error, 
         estimate + 1.96 * standard_error)
ci_99 = (estimate - 2.576 * standard_error, 
         estimate + 2.576 * standard_error)

print("90% CI:", ci_90)
print("95% CI:", ci_95)
print("99% CI:", ci_99)


90% CI: (np.float64(7.584086533098734), np.float64(7.658398436841146))
95% CI: (np.float64(7.576971563591482), np.float64(7.665513406348398))
99% CI: (np.float64(7.563057845443967), np.float64(7.679427124495914))


In [23]:
z = (estimate - 8) / standard_error
p_value = 2 * norm.cdf(-abs(z))

print("z-statistic:", z)
print("p-value:", p_value)

for alpha in [0.10, 0.05, 0.01]:
    print(alpha, "Reject H0" if p_value < alpha else "Fail to reject H0")

print("Difference from 8 hours:", estimate - 8)


z-statistic: -16.76867583379795
p-value: 4.136239445471465e-63
0.1 Reject H0
0.05 Reject H0
0.01 Reject H0
Difference from 8 hours: -0.3787575150300597


Looking at our results above, we can see that the average weekday sleep for adults 18 and up is about 7.62 hours, which is approximately 0.38 hours less than 8 hours. The confidence intervals all exclude 8 hours, which means that the population mean is below 8 hours. Our p-value is extremely small, meaning we can reject the null hypothesis at all three significance levels. Although the difference is statistically significant, 0.38 hour isnt that big of a difference so statistical and practical significance dont really tell the same story.